# Home Pantry Management Portal
**Assignment No. 3 – End-to-End Portal Design**  
**Course:** Programming for Business

**Student name(s):** Fatima Shahid, Rania Babur 
**Roll number(s):** 25L-5033,25L-5090  
**Submitted to:** Ms.Areeba Khan 
**Submission date:** 14 November 2026

## Project overview
A console-based Python portal for recording household grocery items, checking quantities and expiry dates, preparing a shopping lists and estimating pantry value. It demonstrates functions, loops, JSON persistence, input validation and exception handling.



## 1. Import libraries
`json` handles storage, `datetime` handles expiry dates, `math` validates numeric input and `pathlib` handles file paths.

In [2]:
import json, math
from pathlib import Path
from datetime import datetime, date, timedelta

DATA_FILE = Path("pantry_data.json")
DEMO_USERNAME = "fatimarania"
DEMO_PASSWORD = "pantry123"


## 2. Load and save data
Both functions use `try-except-finally`: errors are handled and opened files are closed in the `finally` block.

In [3]:
def load_data():
    file = None
    try:
        if not DATA_FILE.exists():
            return []
        file = open(DATA_FILE, "r", encoding="utf-8")
        data = json.load(file)
        if not isinstance(data, list):
            print("Invalid data format. Starting with an empty pantry.")
            return []
        return data
    except (json.JSONDecodeError, PermissionError, OSError) as error:
        print("Could not read pantry data:", error)
        return []
    finally:
        if file is not None and not file.closed:
            file.close()


def save_data(pantry):
    file = None
    try:
        file = open(DATA_FILE, "w", encoding="utf-8")
        json.dump(pantry, file, indent=4)
        return True
    except (PermissionError, OSError, TypeError) as error:
        print("Could not save pantry data:", error)
        return False
    finally:
        if file is not None and not file.closed:
            file.close()


## 3. User login
The login allows three attempts. It is a simple classroom demonstration, not a secure user-account system.

In [4]:
def user_login():
    print("HOME PANTRY PORTAL LOGIN")
    for attempt in range(3):
        username = input("Username: ").strip()
        password = input("Password: ")
        if username == DEMO_USERNAME and password == DEMO_PASSWORD:
            print("Login successful. Welcome!")
            return True
        remaining = 2 - attempt
        if remaining:
            print(f"Incorrect details. Attempts remaining: {remaining}")
    print("Too many unsuccessful attempts. Portal closed.")
    return False


## 4. Helper functions

In [5]:
def find_item(pantry, item_id):
    for item in pantry:
        if item["id"] == item_id:
            return item
    return None


def get_next_id(pantry):
    return max((item["id"] for item in pantry), default=0) + 1


## 5. Add a pantry item
Required text fields, numbers and optional expiry dates are validated before saving.

In [6]:
def add_item(pantry):
    name = input("Item name: ").strip()
    category = input("Category: ").strip()
    unit = input("Unit (kg/litre/pack/pieces): ").strip()
    if not name or not category or not unit:
        print("Name, category and unit cannot be empty.")
        return False
    try:
        quantity = float(input("Current quantity: "))
        minimum = float(input("Minimum stock level: "))
        price = float(input("Price per unit (PKR): "))
        if not all(math.isfinite(x) and x >= 0 for x in (quantity, minimum, price)):
            print("Enter valid non-negative numbers.")
            return False
    except ValueError:
        print("Please enter numbers for quantity, minimum stock and price.")
        return False
    expiry = input("Expiry date (YYYY-MM-DD, or leave blank): ").strip()
    if expiry:
        try:
            datetime.strptime(expiry, "%Y-%m-%d")
        except ValueError:
            print("Invalid date. Use YYYY-MM-DD.")
            return False
    item = {"id":get_next_id(pantry), "name":name.title(), "category":category.title(),
            "quantity":quantity, "unit":unit.lower(), "expiry_date":expiry,
            "minimum_stock":minimum, "price_per_unit":price}
    pantry.append(item)
    if save_data(pantry):
        print("Item added and saved successfully.")
        return True
    pantry.pop()
    return False


## 6. View pantry items

In [7]:
def view_items(pantry):
    if not pantry:
        print("Your pantry is empty.")
        return
    print("\n--- PANTRY INVENTORY ---")
    for item in pantry:
        print(f'ID: {item["id"]} | {item["name"]} | {item["category"]} | '
              f'{item["quantity"]} {item["unit"]} | Expiry: {item["expiry_date"] or "Not entered"}')


## 7. Search for an item
Search is case-insensitive and accepts part of an item name.

In [8]:
def search_item(pantry):
    term = input("Enter item name to search: ").strip().lower()
    if not term:
        print("Please enter a search term.")
        return []
    matches = [item for item in pantry if term in item["name"].lower()]
    for item in matches:
        print(item)
    if not matches:
        print("No matching item was found.")
    return matches


## 8. Update quantity

In [9]:
def update_quantity(pantry):
    try:
        item_id = int(input("Enter item ID: "))
        quantity = float(input("Enter new quantity: "))
        if not math.isfinite(quantity) or quantity < 0:
            print("Quantity must be a valid non-negative number.")
            return False
    except ValueError:
        print("Please enter a valid ID and numeric quantity.")
        return False
    item = find_item(pantry, item_id)
    if item is None:
        print("Item not found.")
        return False
    old_quantity = item["quantity"]
    item["quantity"] = quantity
    if save_data(pantry):
        print("Quantity updated successfully.")
        return True
    item["quantity"] = old_quantity
    return False


## 9. Delete an item

In [10]:
def delete_item(pantry):
    try:
        item_id = int(input("Enter item ID to delete: "))
    except ValueError:
        print("Please enter a valid item ID.")
        return False
    item = find_item(pantry, item_id)
    if item is None:
        print("Item not found.")
        return False
    pantry.remove(item)
    if save_data(pantry):
        print("Item deleted successfully.")
        return True
    pantry.append(item)
    return False


## 10. Low-stock alert
Items at or below their minimum stock are flagged.

In [11]:
def low_stock_alert(pantry):
    print("\n--- LOW STOCK ITEMS ---")
    low = [item for item in pantry if item["quantity"] <= item["minimum_stock"]]
    for item in low:
        print(f'{item["name"]}: {item["quantity"]} {item["unit"]} remaining.')
    if not low:
        print("No low-stock items.")
    return low


## 11. Expiry check
Shows already expired products and products expiring within the next seven days.

In [12]:
def expiry_check(pantry):
    today = date.today()
    limit = today + timedelta(days=7)
    flagged = []
    print("\n--- EXPIRY CHECK ---")
    for item in pantry:
        if not item["expiry_date"]:
            continue
        try:
            expiry = datetime.strptime(item["expiry_date"], "%Y-%m-%d").date()
            if expiry < today:
                print(f'{item["name"]} is EXPIRED ({expiry}).')
                flagged.append(item)
            elif expiry <= limit:
                print(f'{item["name"]} will expire soon ({expiry}).')
                flagged.append(item)
        except ValueError:
            print(f'Invalid expiry date stored for {item["name"]}.')
    if not flagged:
        print("No expired or soon-to-expire items.")
    return flagged


## 12. Automatic shopping list

In [13]:
def shopping_list(pantry):
    print("\n--- SHOPPING LIST ---")
    items = low_stock_alert(pantry)
    if not items:
        print("Your shopping list is empty.")
    return items


## 13. Calculate pantry value

In [14]:
def pantry_value(pantry):
    total = sum(item["quantity"] * item["price_per_unit"] for item in pantry)
    print("Estimated pantry value: PKR", round(total, 2))
    return total


## 14. Generate pantry report
Quantities in different units are not combined into one physical total.

In [15]:
def generate_report(pantry):
    categories = []
    for item in pantry:
        if item["category"] not in categories:
            categories.append(item["category"])
    low_count = sum(1 for item in pantry if item["quantity"] <= item["minimum_stock"])
    value = pantry_value(pantry)
    print("\n======= PANTRY SUMMARY REPORT =======")
    print("Different products:", len(pantry))
    print("Categories:", ", ".join(categories) if categories else "None")
    print("Low-stock products:", low_count)
    print("Estimated value: PKR", round(value, 2))
    print("=====================================")
    return {"product_count":len(pantry), "categories":categories,
            "low_stock_count":low_count, "estimated_value":round(value,2)}


## 15. Automated sample tests


In [16]:
from unittest.mock import patch
from tempfile import TemporaryDirectory
from contextlib import redirect_stdout
import io

def run_sample_tests():
    global DATA_FILE
    original_file = DATA_FILE
    output = io.StringIO()
    with TemporaryDirectory() as temp:
        DATA_FILE = Path(temp) / "test_pantry.json"
        pantry_test = []
        with redirect_stdout(output):
            with patch("builtins.input", side_effect=[
                "Test Rice", "Grains", "kg", "0.5", "1", "300",
                (date.today() + timedelta(days=3)).strftime("%Y-%m-%d")
            ]):
                added = add_item(pantry_test)
            with patch("builtins.input", return_value="rice"):
                found = search_item(pantry_test)
            with patch("builtins.input", side_effect=["1", "2.0"]):
                updated = update_quantity(pantry_test)
            pantry_test[0]["quantity"] = 0.5
            low = low_stock_alert(pantry_test)
            expiry = expiry_check(pantry_test)
            summary = generate_report(pantry_test)
            with patch("builtins.input", return_value="1"):
                deleted = delete_item(pantry_test)
        checks = {
            "Add item and save": added is True,
            "Search existing item": len(found) == 1,
            "Update quantity": updated is True,
            "Low-stock alert": len(low) == 1,
            "Expiry alert": len(expiry) == 1,
            "Generate summary report": summary["product_count"] == 1,
            "Delete item": deleted is True and len(pantry_test) == 0,
            "JSON persistence": DATA_FILE.exists()
        }
    DATA_FILE = original_file
    for label, passed in checks.items():
        print(f"{label}: {'PASS' if passed else 'FAIL'}")
    if not all(checks.values()):
        raise AssertionError("One or more sample tests failed.")

run_sample_tests()


Add item and save: PASS
Search existing item: PASS
Update quantity: PASS
Low-stock alert: PASS
Expiry alert: PASS
Generate summary report: PASS
Delete item: PASS
JSON persistence: PASS


## 16. Main menu
The `while True` loop continues until the user chooses Exit. Invalid menu choices are handled.

In [17]:
def main():
    pantry = load_data()
    if not user_login():
        print("Portal closed.")
        return
    while True:
        print("\n========== HOME PANTRY PORTAL ==========")
        print("1. Add new item")
        print("2. View all items")
        print("3. Search item")
        print("4. Update quantity")
        print("5. Delete item")
        print("6. Low-stock alert")
        print("7. Expiry check")
        print("8. Automatic shopping list")
        print("9. Calculate pantry value")
        print("10. Generate pantry report")
        print("11. Exit")
        choice = input("Choose an option: ").strip()
        if choice == "1": add_item(pantry)
        elif choice == "2": view_items(pantry)
        elif choice == "3": search_item(pantry)
        elif choice == "4": update_quantity(pantry)
        elif choice == "5": delete_item(pantry)
        elif choice == "6": low_stock_alert(pantry)
        elif choice == "7": expiry_check(pantry)
        elif choice == "8": shopping_list(pantry)
        elif choice == "9": pantry_value(pantry)
        elif choice == "10": generate_report(pantry)
        elif choice == "11":
            save_data(pantry)
            print("Thank you for using Home Pantry Portal.")
            break
        else:
            print("Invalid choice. Select a number from 1 to 11.")


# Change to True to start the interactive menu.
RUN_INTERACTIVE_MENU = True
if RUN_INTERACTIVE_MENU:
    main()
else:
    print("Menu is ready. Set RUN_INTERACTIVE_MENU = True to start.")


HOME PANTRY PORTAL LOGIN


Username:  fatimarania
Password:  pantry12


Incorrect details. Attempts remaining: 2


Username:  pantry123
Password:  pantry


Incorrect details. Attempts remaining: 1


Username:  fatimarania
Password:  pantry123


... (21 outputs left out)

Choose an option:  10


Estimated pantry value: PKR 270.0

======= PANTRY SUMMARY REPORT =======
Different products: 2
Categories: Cake, Brownies
Low-stock products: 2
Estimated value: PKR 270.0

========== HOME PANTRY PORTAL ==========
1. Add new item
2. View all items
3. Search item
4. Update quantity
5. Delete item
6. Low-stock alert
7. Expiry check
8. Automatic shopping list
9. Calculate pantry value
10. Generate pantry report
11. Exit


Choose an option:  11


Thank you for using Home Pantry Portal.
